# SiMaster vs NaMaster

Side-by-side comparison of QML (SiMaster) and pseudo-$C_\ell$ (NaMaster)
bandpower estimation on the same simulated data: a spin-0 CMB-like field at
`nside=16` with the NaMaster test-suite mask, 100 realizations.

The point of QML is the large scales: exact mode decoupling and optimal
(smaller) error bars where the mask couples modes the most.

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import numpy as np, healpy as hp
import matplotlib.pyplot as plt
import pymaster as nmt
import simaster as sm

nside, lmax, nlb, nreal = 16, 47, 4, 100
npix = hp.nside2npix(nside)
rng = np.random.default_rng(1)

# NaMaster test-suite inputs (mask + CMB TT spectrum)
root = os.path.join("..", "..", "resources", "code", "NaMaster", "test")
mask = (hp.ud_grade(hp.read_map(os.path.join(root, "mask.fits")), nside) > 0.5).astype(float)
ell, cltt, clee, clbb, clte = np.loadtxt(os.path.join(root, "cls.txt"), unpack=True)
cl = cltt[:lmax+1].copy(); cl[:2] = 0
sigma_pix = 30.0                      # uK noise per pixel
ivar = np.full(npix, 1/sigma_pix**2)
print(f"fsky = {mask.mean():.3f}")

In [ ]:
# 100 common realizations
maps = np.array([hp.synfast(cl, nside, lmax=lmax) * mask
                 + rng.normal(0, sigma_pix, npix) for _ in range(nreal)])

## NaMaster (pseudo-$C_\ell$)

In [ ]:
b_nmt = nmt.NmtBin.from_nside_linear(nside, nlb)
f0 = nmt.NmtField(mask, [maps[0]])
w_nmt = nmt.NmtWorkspace.from_fields(f0, f0, b_nmt)
nl_bias = b_nmt.bin_cell(np.full((1, 3*nside), sigma_pix**2 * hp.nside2pixarea(nside)))
cl_nmt = []
for m in maps:
    f = nmt.NmtField(mask, [m])
    cl_nmt.append(w_nmt.decouple_cell(nmt.compute_coupled_cell(f, f))[0])
cl_nmt = np.array(cl_nmt) - nl_bias[0]
ells_n = b_nmt.get_effective_ells()

## SiMaster (QML)

Same data; the fiducial spectrum is the truth here (you can perturb it --
estimates stay unbiased, only error bars react). The exact response engine
gives the Fisher matrix, noise bias and window functions deterministically.

In [ ]:
b_sm = sm.Bins.linear(2, lmax, nlb)
fld = sm.Field(mask, [np.zeros(npix)], ivar=ivar, name="T")
w_sm = sm.QMLWorkspace(fld, b_sm, {("T_0", "T_0"): cl}, lmax=lmax,
                       fisher_mode="exact", verbose=False)
res = w_sm.estimate(w_sm.pack_data([maps]))
cl_qml = res.cl["T_0 x T_0"]
ells_q = res.ells
err_qml = np.sqrt(np.diag(res.cov))          # QML predicted errors

In [ ]:
fac_q = ells_q*(ells_q+1)/2/np.pi
fac_n = ells_n*(ells_n+1)/2/np.pi
plt.figure(figsize=(7,4.5))
lt = np.arange(2, lmax+1)
plt.plot(lt, lt*(lt+1)/2/np.pi*cl[2:], 'k-', lw=1, label='input')
plt.errorbar(ells_q-0.4, fac_q*cl_qml.mean(0), fac_q*cl_qml.std(0), fmt='o', ms=4,
             label='SiMaster QML (mean ± std over 100)')
plt.errorbar(ells_n+0.4, fac_n*cl_nmt.mean(0), fac_n*cl_nmt.std(0), fmt='s', ms=4,
             label='NaMaster pCl')
plt.xlabel(r'$\ell$'); plt.ylabel(r'$\ell(\ell+1)C_\ell/2\pi\ [\mu K^2]$')
plt.legend(); plt.title('Same 100 masked+noisy realizations'); plt.tight_layout()

In [ ]:
# Error-bar comparison: empirical scatter of both estimators
plt.figure(figsize=(7,4))
plt.plot(ells_q, cl_qml.std(0)/cl_nmt.std(0)[:len(ells_q)], 'o-',
         label='QML std / pCl std (empirical)')
plt.plot(ells_q, err_qml/cl_qml.std(0), 's--',
         label='QML predicted / empirical')
plt.axhline(1, color='k', lw=0.5)
plt.xlabel(r'$\ell$'); plt.legend(); plt.tight_layout()
print('QML/pCl error ratio per band:', np.round(cl_qml.std(0)/cl_nmt.std(0)[:len(ells_q)], 3))

The QML error bars are visibly tighter in the lowest bands (where the
mask couples modes) and the predicted covariance matches the empirical
scatter. At high $\ell$ the two estimators converge, which is why a hybrid
QML (low $\ell$) + NaMaster (high $\ell$) analysis is the practical
recipe -- see `docs/migration.md`.